# Food Waste Analysis - Kafka/Spark Results

Run this notebook first to create or refresh the `food_waste.country_continent` and `food_waste.activity_stage_map` lookup tables in PostgreSQL. Spark reads observations from the `food-waste-raw` Kafka topic and reads both lookup tables directly from PostgreSQL through JDBC; no lookup CSV files are used. The Kafka consumer writes Spark outputs to PostgreSQL.

Then run `spark/jobs/food_waste_processing_main.py`. Finally, rerun the **Gold analysis tables** and **Dashboard CSV** cells below to display the new outputs.

The setup uses the existing PostgreSQL source tables and the setup/views block in `db/queries/food_waste_analysis.sql`.

In [ ]:

from pathlib import Path
import os
import re
import pandas as pd
import psycopg2
from dotenv import load_dotenv
from sqlalchemy.engine import make_url
from IPython.display import display

HERE = Path.cwd().resolve()
PROJECT_ROOT = HERE.parent if HERE.name.lower() == "notebooks" else HERE

SQL_FILE = PROJECT_ROOT / "db" / "queries" / "food_waste_analysis.sql"
OUT = PROJECT_ROOT / "data" / "processed_by_spark"
GOLD_DIR = Path(OUT) / "gold"
DASHBOARD_CSV = Path(OUT) / "dashboard" / "food_waste_dashboard.csv"

# Load environment variables
load_dotenv(PROJECT_ROOT / ".env")

DATABASE_URL = os.getenv("DATABASE_URL")

if not DATABASE_URL:
    raise RuntimeError("DATABASE_URL not found in environment variables.")

db_url = make_url(DATABASE_URL)

# Detect where the notebook is running
RUNNING_IN_DOCKER = (
    os.getenv("RUNNING_IN_DOCKER", "false").lower() == "true"
)

db_host = db_url.host

if RUNNING_IN_DOCKER:
    # Airflow / Docker containers communicate using service names
    if db_host in ("localhost", "127.0.0.1"):
        db_host = "postgres"
else:
    # Windows host communicates through the published Docker port
    if db_host == "postgres":
        db_host = "localhost"

DB_CONFIG = {
    "host": db_host,
    "port": db_url.port or 5432,
    "dbname": db_url.database,
    "user": db_url.username,
    "password": db_url.password,
}

if not SQL_FILE.exists():
    raise FileNotFoundError(f"Setup SQL not found: {SQL_FILE}")

print(f"Project: {PROJECT_ROOT}")
print(f"Database: {DB_CONFIG['dbname']}")
print(f"Host: {DB_CONFIG['host']}:{DB_CONFIG['port']}")
print(f"Running in Docker: {RUNNING_IN_DOCKER}")


Project: D:\My_projects\data-engineer\waste-food-project
Database: food_waste_db
Host: localhost:5432


## 1. Create lookup tables for Spark in PostgreSQL

This cell executes `db/queries/food_waste_analysis.sql` to create the lookup tables in PostgreSQL. Spark connects to these tables directly; this notebook does not export lookup CSV files.

In [4]:
sql_text = SQL_FILE.read_text(encoding="utf-8-sig")
setup_sql = re.sub(r"(?m)^--\s*(?:name|page|title|question|visual):.*(?:\n|$)", "", sql_text).strip()
connection = psycopg2.connect(**DB_CONFIG)
try:
    with connection.cursor() as cursor:
        cursor.execute(setup_sql)
    connection.commit()

    country_lookup = pd.read_sql_query(
        "SELECT LTRIM(m49_code, '0') AS m49_code, country_name, continent "
        "FROM food_waste.country_continent ORDER BY continent, country_name",
        connection,
        dtype={"m49_code": str},
    )
    activity_lookup = pd.read_sql_query(
        "SELECT activity_name, stage_group FROM food_waste.activity_stage_map "
        "ORDER BY activity_name",
        connection,
    )


    print(f"food_waste.country_continent: {len(country_lookup):,} rows")
    print(f"food_waste.activity_stage_map: {len(activity_lookup):,} rows")
finally:
    connection.close()

food_waste.country_continent: 142 rows
food_waste.activity_stage_map: 52 rows


C:\Users\GM\AppData\Local\Temp\ipykernel_38816\2525637384.py:9: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  country_lookup = pd.read_sql_query(
C:\Users\GM\AppData\Local\Temp\ipykernel_38816\2525637384.py:15: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  activity_lookup = pd.read_sql_query(


## 2. Review the lookup tables read from PostgreSQL

In [5]:
lookups = {
    "country_continent": country_lookup,
    "activity_stage_map": activity_lookup,
}
for name, frame in lookups.items():
    print(f"{name}: {len(frame):,} rows")
    display(frame.head(10))

country_continent: 142 rows


,m49_code,country_name,continent
0,12,Algeria,Africa
1,24,Angola,Africa
2,204,Benin,Africa
3,72,Botswana,Africa
4,854,Burkina Faso,Africa
5,108,Burundi,Africa
6,120,Cameroon,Africa
7,148,Chad,Africa
8,180,DR Congo,Africa
9,818,Egypt,Africa


activity_stage_map: 52 rows


,activity_name,stage_group
0,Assembling,Post-harvest handling
1,Bagging,Post-harvest handling
2,Blanching,Processing
3,Bulking,Post-harvest handling
4,Bundling,Post-harvest handling
5,Cleaning,Post-harvest handling
6,Collection,Transport & distribution
7,Consumption,Consumption
8,Curing,Post-harvest handling
9,Dewatering,Post-harvest handling


## 3. Gold analysis tables

Run `food_waste_processing_main.py` after creating the lookups, then run this cell to load and display the new Gold CSV tables.

In [6]:
tables = {}
if not GOLD_DIR.exists():
    print(f"Gold outputs are not present yet: {GOLD_DIR}. Run the Spark processing job, then rerun this cell.")
else:
    for table_dir in sorted(path for path in GOLD_DIR.iterdir() if path.is_dir()):
        part_files = sorted(table_dir.glob("part-*.csv"))
        if not part_files:
            continue
        frames = [pd.read_csv(file) for file in part_files]
        tables[table_dir.name] = pd.concat(frames, ignore_index=True) if len(frames) > 1 else frames[0]
        print(f"{table_dir.name}: {len(tables[table_dir.name]):,} rows")
        display(tables[table_dir.name].head(20))

activity_count_profile: 11 rows


,activity_count,observations,percentage
0,1,23365,74.97
1,2,7479,24.00
2,3,110,0.35
3,4,34,0.11
4,5,9,0.03
5,6,7,0.02
6,7,7,0.02
7,8,10,0.03
8,9,1,0.00
9,10,140,0.45


continent: 5 rows


,continent,crops,records,countries,avg_loss_pct,median_loss_pct,p90_loss_pct
0,Oceania,25,115,4,10.80,8.80,26.60
1,Americas,142,3466,25,10.57,7.94,26.63
2,Asia,108,4204,37,5.82,3.10,15.00
3,Africa,83,21330,45,3.48,2.50,5.60
4,Europe,83,2049,31,4.06,2.10,11.00


continent_year: 85 rows


,continent,year,records,countries,avg_loss_pct,median_loss_pct,p90_loss_pct
0,Africa,2000,866,40,2.77,2.50,4.52
1,Africa,2001,867,40,2.74,2.50,4.54
2,Africa,2002,875,40,2.88,2.50,4.55
3,Africa,2003,866,40,2.83,2.50,4.55
4,Africa,2004,867,40,2.80,2.50,4.55
5,Africa,2005,876,41,3.03,2.50,4.55
6,Africa,2006,879,40,3.00,2.50,4.55
7,Africa,2007,971,40,3.17,2.50,4.55
8,Africa,2008,1013,40,2.99,2.50,4.78
9,Africa,2009,987,40,4.48,2.65,8.12


... (48 outputs left out)

yearly: 27 rows


,year,records,countries,commodities,avg_loss_pct,p25_loss_pct,median_loss_pct,p75_loss_pct,is_reliable_year
0,2000,1259,83,124,4.23,1.41,2.65,4.40,True
1,2001,1417,84,132,4.58,1.64,2.65,4.50,True
2,2002,1341,83,129,4.64,1.65,2.65,4.55,True
3,2003,1288,84,126,4.66,1.58,2.65,4.43,True
4,2004,1221,83,119,4.11,1.36,2.63,4.40,True
5,2005,1303,84,117,5.06,1.65,2.65,4.55,True
6,2006,1702,78,127,3.99,1.14,2.50,4.35,True
7,2007,1291,83,112,4.13,1.50,2.65,4.40,True
8,2008,1391,82,111,5.26,1.36,2.65,4.59,True
9,2009,1712,76,97,4.79,1.55,2.69,4.55,True


yearly_trend: 22 rows


,year,records,countries,commodities,avg_loss_pct,p25_loss_pct,median_loss_pct,p75_loss_pct,is_reliable_year,change_vs_previous_reliable_year
0,2000,1259,83,124,4.23,1.41,2.65,4.40,True,NaN
1,2001,1417,84,132,4.58,1.64,2.65,4.50,True,0.00
2,2002,1341,83,129,4.64,1.65,2.65,4.55,True,0.00
3,2003,1288,84,126,4.66,1.58,2.65,4.43,True,0.00
4,2004,1221,83,119,4.11,1.36,2.63,4.40,True,-0.02
5,2005,1303,84,117,5.06,1.65,2.65,4.55,True,0.02
6,2006,1702,78,127,3.99,1.14,2.50,4.35,True,-0.15
7,2007,1291,83,112,4.13,1.50,2.65,4.40,True,0.15
8,2008,1391,82,111,5.26,1.36,2.65,4.59,True,0.00
9,2009,1712,76,97,4.79,1.55,2.69,4.55,True,0.04


## 4. Dashboard CSV

In [7]:
if DASHBOARD_CSV.exists():
    dashboard = pd.read_csv(DASHBOARD_CSV)
    print(f"Dashboard: {len(dashboard):,} rows, {len(dashboard.columns)} columns")
    display(dashboard.head(20))
else:
    dashboard = None
    print(f"Dashboard CSV not found yet: {DASHBOARD_CSV}. Run the Spark job, then rerun this cell.")

Dashboard: 25,617 rows, 23 columns


,country,continent,m49_code,commodity,product_group,cpc_code,year,decade,time_period,is_reliable_year,...,method_group,is_modelled,loss_percentage,median_loss,min_loss,max_loss,loss_sum,count,loss_band,is_merged_group
0,Afghanistan,Asia,4,Barley,Cereals,0115,2014,2010s,2011-2021,True,...,"Measured, surveyed or published",False,14.78,14.78,14.78,14.78,14.78,1,4. 10 to 20,False
1,Afghanistan,Asia,4,Barley,Cereals,0115,2015,2010s,2011-2021,True,...,"Measured, surveyed or published",False,14.89,14.89,14.89,14.89,14.89,1,4. 10 to 20,False
2,Afghanistan,Asia,4,Barley,Cereals,0115,2016,2010s,2011-2021,True,...,"Measured, surveyed or published",False,14.58,14.58,14.58,14.58,14.58,1,4. 10 to 20,False
3,Afghanistan,Asia,4,Barley,Cereals,0115,2017,2010s,2011-2021,True,...,"Measured, surveyed or published",False,14.74,14.74,14.74,14.74,14.74,1,4. 10 to 20,False
4,Afghanistan,Asia,4,Maize (corn),Cereals,0112,2014,2010s,2011-2021,True,...,"Measured, surveyed or published",False,14.87,14.87,14.87,14.87,14.87,1,4. 10 to 20,False
5,Afghanistan,Asia,4,Maize (corn),Cereals,0112,2015,2010s,2011-2021,True,...,"Measured, surveyed or published",False,14.87,14.87,14.87,14.87,14.87,1,4. 10 to 20,False
6,Afghanistan,Asia,4,Maize (corn),Cereals,0112,2016,2010s,2011-2021,True,...,"Measured, surveyed or published",False,15.08,15.08,15.08,15.08,15.08,1,4. 10 to 20,False
7,Afghanistan,Asia,4,Maize (corn),Cereals,0112,2017,2010s,2011-2021,True,...,"Measured, surveyed or published",False,14.95,14.95,14.95,14.95,14.95,1,4. 10 to 20,False
8,Afghanistan,Asia,4,Rice,Cereals,0113,2014,2010s,2011-2021,True,...,"Measured, surveyed or published",False,6.70,6.70,6.70,6.70,6.70,1,3. 5 to 10,False
9,Afghanistan,Asia,4,Rice,Cereals,0113,2015,2010s,2011-2021,True,...,"Measured, surveyed or published",False,7.07,7.07,7.07,7.07,7.07,1,3. 5 to 10,False
